In [1]:
from pyspark.sql import functions as F

# ── Read Bronze sources ───────────────────────────────────────────────
df_product = spark.read.table("lh_adventureworks_bronze.dbo.Production_Product")
df_subcat = spark.read.table("lh_adventureworks_bronze.dbo.Production_ProductSubcategory")
df_cat = spark.read.table("lh_adventureworks_bronze.dbo.Production_ProductCategory")

print("Product rows      :", df_product.count())
print("Subcategory rows  :", df_subcat.count())
print("Category rows     :", df_cat.count())

StatementMeta(, 13815b1f-f8f3-436f-b6e5-5bb8fa26e4d7, 3, Finished, Available, Finished, False)

Product rows      : 504
Subcategory rows  : 37
Category rows     : 4


In [2]:
# ── Join Product + Subcategory + Category ─────────────────────────────
df_subcat_clean = df_subcat.select(
    F.col("ProductSubcategoryID").alias("sc_ProductSubcategoryID"),
    F.col("ProductCategoryID").alias("sc_ProductCategoryID"),
    F.col("Name").alias("SubcategoryName"),
    F.col("rowguid").alias("sc_rowguid"),
    F.col("ModifiedDate").alias("sc_ModifiedDate")
)

df_cat_clean = df_cat.select(
    F.col("ProductCategoryID").alias("c_ProductCategoryID"),
    F.col("Name").alias("CategoryName"),
    F.col("rowguid").alias("c_rowguid"),
    F.col("ModifiedDate").alias("c_ModifiedDate")
)

# Join product → subcategory
df_silver_product = df_product.join(
    df_subcat_clean,
    df_product["ProductSubcategoryID"] == df_subcat_clean["sc_ProductSubcategoryID"],
    how="left"
).drop("sc_ProductSubcategoryID")

# Join → category
df_silver_product = df_silver_product.join(
    df_cat_clean,
    df_silver_product["sc_ProductCategoryID"] == df_cat_clean["c_ProductCategoryID"],
    how="left"
).drop("c_ProductCategoryID")

# ── Clean up and select final columns ─────────────────────────────────
df_silver_product = df_silver_product.select(
    "ProductID",
    "Name",
    F.col("ProductNumber"),
    F.col("Color"),
    F.col("Size"),
    F.col("Weight"),
    F.col("StandardCost"),
    F.col("ListPrice"),
    F.col("SellStartDate"),
    F.col("SellEndDate"),
    F.col("DiscontinuedDate"),
    F.col("ProductSubcategoryID"),
    F.col("SubcategoryName"),
    F.col("sc_ProductCategoryID").alias("ProductCategoryID"),
    F.col("CategoryName"),
    F.col("ProductLine"),
    F.col("Class"),
    F.col("Style"),
    F.col("SafetyStockLevel"),
    F.col("ReorderPoint"),
    F.col("DaysToManufacture"),
    F.col("rowguid"),
    F.col("ModifiedDate")
)

# ── Type standardization ──────────────────────────────────────────────
df_silver_product = df_silver_product \
    .withColumn("StandardCost", F.col("StandardCost").cast("decimal(19,4)")) \
    .withColumn("ListPrice", F.col("ListPrice").cast("decimal(19,4)")) \
    .withColumn("Weight", F.col("Weight").cast("decimal(8,2)"))


StatementMeta(, 13815b1f-f8f3-436f-b6e5-5bb8fa26e4d7, 4, Finished, Available, Finished, False)

In [3]:
# ── DQ checks ─────────────────────────────────────────────────────────
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("silver.product.dq")

dq_passed = True
dq_results = []

def log_check(name, passed, message):
    global dq_passed
    status = "PASS" if passed else "FAIL"
    log_msg = f"[DQ] [{status}] {name} — {message}"
    if passed:
        logger.info(log_msg)
    else:
        logger.error(log_msg)
        dq_passed = False
    dq_results.append({"check": name, "status": status, "message": message})

actual_rows = df_silver_product.count()

log_check(
    "Row count",
    actual_rows == df_product.count(),
    f"Expected {df_product.count():,}, got {actual_rows:,}"
)

null_pk = df_silver_product.filter(F.col("ProductID").isNull()).count()
log_check("Null PK — ProductID", null_pk == 0, f"{null_pk:,} nulls")

dup_pk = actual_rows - df_silver_product.select("ProductID").distinct().count()
log_check("Duplicate PK — ProductID", dup_pk == 0, f"{dup_pk:,} duplicates")

unmatched_subcat = df_silver_product.filter(
    F.col("ProductSubcategoryID").isNotNull() &
    F.col("SubcategoryName").isNull()
).count()
log_check(
    "Subcategory join coverage",
    unmatched_subcat == 0,
    f"{unmatched_subcat:,} products with SubcategoryID but no SubcategoryName"
)

failed_checks = [r for r in dq_results if r["status"] == "FAIL"]
if failed_checks:
    failed_names = ", ".join([r["check"] for r in failed_checks])
    raise Exception(
        f"silver.product DQ failed — {len(failed_checks)} check(s) failed: "
        f"{failed_names}. Pipeline halted."
    )

logger.info(f"[DQ] All checks passed — {actual_rows:,} rows cleared for Silver write.")

StatementMeta(, 13815b1f-f8f3-436f-b6e5-5bb8fa26e4d7, 5, Finished, Available, Finished, False)

INFO:silver.product.dq:[DQ] [PASS] Row count — Expected 504, got 504
INFO:silver.product.dq:[DQ] [PASS] Null PK — ProductID — 0 nulls
INFO:silver.product.dq:[DQ] [PASS] Duplicate PK — ProductID — 0 duplicates
INFO:silver.product.dq:[DQ] [PASS] Subcategory join coverage — 0 products with SubcategoryID but no SubcategoryName
INFO:silver.product.dq:[DQ] All checks passed — 504 rows cleared for Silver write.


In [4]:
# ── Create schema and write ───────────────────────────────────────────
#spark.sql("CREATE SCHEMA IF NOT EXISTS lh_adventureworks_silver.production")

df_silver_product.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("lh_adventureworks_silver.production.product")

StatementMeta(, 13815b1f-f8f3-436f-b6e5-5bb8fa26e4d7, 6, Finished, Available, Finished, False)

In [5]:
# ── Verify ────────────────────────────────────────────────────────────
df_verify = spark.read.table("lh_adventureworks_silver.production.product")
print(f"silver.production.product written — {df_verify.count():,} rows verified.")

StatementMeta(, 13815b1f-f8f3-436f-b6e5-5bb8fa26e4d7, 7, Finished, Available, Finished, False)

silver.production.product written — 504 rows verified.
